# STDP 学习规则更新权重

In [1]:
from network import *
from optimizer import STDP
from functools import partial

In [2]:
def i_fn(t, freq, duration, amplitude):
    if t % freq < duration:
        return amplitude
    return 0


current_fn1 = partial(i_fn, freq=12, duration=3, amplitude=10)
current_fn2 = partial(i_fn, freq=10, duration=3, amplitude=10)
current_fn3 = partial(i_fn, freq=8, duration=3, amplitude=10)

In [3]:
T = 1000
dt = 0.1
n_t = int(T / dt) + 1
time = np.arange(0, T + dt, dt)

network = Network([
    NeuronGroup(3, l=1000, d=10.0, r_a=25.0),
    NeuronGroup(10, l=1000, d=10.0, r_a=25.0),
    NeuronGroup(2, l=1000, d=10.0, r_a=25.0),
], g=0.4, e_syn=-65.0, delay=0.5)
network.add_current_injector(current_fn1, delay=0.5)
network.add_current_injector(current_fn2, delay=0.5)
network.add_current_injector(current_fn3, delay=0.5)
network.update_recoder(1000)

f_weight = lambda x: np.clip(x, -1, 1)
optimizer = STDP(network.synapses, f_pre=f_weight, f_post=f_weight)

In [ ]:
epochs = 100
weights1 = [network.synapses[0].synapses[0].weight]
weights2 = [network.synapses[0].synapses[10].weight]
weights3 = [network.synapses[0].synapses[20].weight]
for i in range(epochs):
    for t in tqdm(range(n_t), desc="Time"):
        network.step(dt)
        optimizer.step()
        weights1.append(network.synapses[0].synapses[0].weight)
        weights2.append(network.synapses[0].synapses[10].weight)
        weights3.append(network.synapses[0].synapses[20].weight)
    optimizer.apply()

Time:  42%|████▏     | 4154/10001 [00:40<00:56, 104.37it/s]